# 8.2 Sentiment Analysis

Classify IMDb reviews as negative (`0`) or positive (`1`). Implement basic normalization, train a supplied baseline, and design two independent alternatives. Compare their test accuracy and classify future texts.

**Add new code cells immediately below each implementation prompt, before its checks. Preserve all supplied cells.** Add markdown explanations for both alternatives and your comparison. See README.md for setup, dataset download, contracts, and scoring. No grid search, cross-validation, or minimum accuracy is required.


In [ ]:
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.base import clone
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline

# Add imports needed by your chosen NLP techniques in your own cells.


## Supplied dataset and partition

Download the complete IMDb archive from https://ai.stanford.edu/~amaas/data/sentiment/ and extract it into this activity's `data/` directory, preserving `aclImdb/`. You may copy the complete download from 7.1. See README.md.

The loader selects 8,000 official training reviews (4,000 per class) and 2,000 official test reviews (1,000 per class). For each partition separately, sorted filenames and a fresh `random.Random(42)` fix selection and order. Common preparation strips HTML tags. Labels are `0 = negative`, `1 = positive`.

Keep all supplied data unchanged. Do not split again, shuffle, filter, or learn vocabulary/IDF from test data. All three pipelines must use exactly the supplied pandas Series.


In [ ]:
activity_root = Path.cwd()
if activity_root.name == "notebooks":
    activity_root = activity_root.parent
dataset_root = activity_root / "data" / "aclImdb"


def load_partition(root: Path, partition: str, per_class: int) -> pd.DataFrame:
    rng = random.Random(42)
    selected = []
    for sentiment, label in (("pos", 1), ("neg", 0)):
        directory = root / partition / sentiment
        candidates = sorted(directory.glob("*.txt"))
        if len(candidates) != 12500:
            raise FileNotFoundError(
                f"Expected 12,500 reviews in {directory}. "
                "Download and extract the complete IMDb dataset; see README.md."
            )
        selected.extend((path, label) for path in rng.sample(candidates, per_class))
    rng.shuffle(selected)
    return pd.DataFrame([
        {
            "review_id": path.relative_to(root).as_posix(),
            "text": re.sub(r"<[^>]+>", " ", path.read_text(encoding="utf-8")),
            "label": label,
        }
        for path, label in selected
    ])


train_reviews = load_partition(dataset_root, "train", 4000)
test_reviews = load_partition(dataset_root, "test", 1000)
X_train = train_reviews["text"].copy()
X_test = test_reviews["text"].copy()
y_train = train_reviews["label"].copy()
y_test = test_reviews["label"].copy()
_fixed_inputs = tuple(series.copy(deep=True) for series in (X_train, X_test, y_train, y_test))
print("Training reviews:", len(train_reviews), "Test reviews:", len(test_reviews))
print("Training labels:", y_train.value_counts().sort_index().to_dict())
print("Test labels:", y_test.value_counts().sort_index().to_dict())
print(train_reviews[["review_id", "label"]].head().to_string(index=False))


## Supplied checking helpers

These helpers support the scored assertion cells. They check unchanged inputs, model settings, predictions, calculated accuracy, and fitting against a fresh copy trained only on the supplied training set. Refitting copies takes additional time. Preserve these helpers.

Checks do not require a specific alternative vocabulary or score. Teacher review also confirms the advanced technique and explanations.


In [ ]:
def _check_fixed_inputs():
    for actual, expected in zip((X_train, X_test, y_train, y_test), _fixed_inputs):
        pd.testing.assert_series_equal(actual, expected)


def _check_classifier(pipeline):
    assert isinstance(pipeline, Pipeline), "Create a sklearn Pipeline instance."
    assert "vectorizer" in pipeline.named_steps
    assert pipeline.steps[-1][0] == "classifier"
    classifier = pipeline.named_steps["classifier"]
    assert isinstance(classifier, LogisticRegression)
    assert classifier.get_params() == LogisticRegression(max_iter=1000).get_params()
    vectorizer = pipeline.named_steps["vectorizer"]
    assert isinstance(vectorizer, (CountVectorizer, TfidfVectorizer))
    return vectorizer, classifier


def _check_fitted_pipeline(pipeline, predictions, accuracy):
    _check_fixed_inputs()
    vectorizer, classifier = _check_classifier(pipeline)
    assert isinstance(predictions, np.ndarray)
    assert predictions.shape == (len(X_test),)
    assert set(predictions.tolist()).issubset({0, 1})
    np.testing.assert_array_equal(predictions, pipeline.predict(X_test))
    assert np.isscalar(accuracy) and 0.0 <= accuracy <= 1.0
    np.testing.assert_allclose(accuracy, accuracy_score(y_test, predictions), rtol=0, atol=1e-12)
    matrix = pipeline[:-1].transform(X_test)
    assert sparse.issparse(matrix), "Keep document-term matrices sparse."
    assert matrix.shape[0] == len(X_test)
    np.testing.assert_array_equal(classifier.classes_, [0, 1])

    # Teacher check: refit the declared design on training data only.
    reference = clone(pipeline)
    reference.fit(_fixed_inputs[0], _fixed_inputs[2])
    reference_vectorizer = reference.named_steps["vectorizer"]
    reference_classifier = reference.named_steps["classifier"]
    assert vectorizer.vocabulary_ == reference_vectorizer.vocabulary_
    if isinstance(vectorizer, TfidfVectorizer) and vectorizer.use_idf:
        np.testing.assert_allclose(vectorizer.idf_, reference_vectorizer.idf_)
    np.testing.assert_allclose(classifier.coef_, reference_classifier.coef_, rtol=1e-5, atol=1e-7)
    np.testing.assert_allclose(classifier.intercept_, reference_classifier.intercept_, rtol=1e-5, atol=1e-7)


def _check_independent(pipeline, others):
    for other in others:
        assert pipeline is not other, "Construct a new Pipeline instance."
        for step in ("vectorizer", "classifier"):
            assert pipeline.named_steps[step] is not other.named_steps[step], (
                f"Each pipeline needs its own new {step} instance."
            )
        assert (
            [(type(step), step.get_params(deep=False)) for _, step in pipeline.steps[:-1]]
            != [(type(step), step.get_params(deep=False)) for _, step in other.steps[:-1]]
        ), "Change preprocessing or vectorization, not just the variable name."


## 1. Baseline preprocessing — 20 points

**Add a new code cell immediately below this cell** defining `normalize_text(text: str) -> str`.

Lowercase the text; replace every character other than `a`–`z` or whitespace with a space; collapse whitespace and trim both ends. Empty or entirely non-alphabetic input returns `""`. Do not stem, lemmatize, or remove stopwords.

Examples: `"GREAT movie!!! 10/10"` → `"great movie"`; `"good-bad"` → `"good bad"`.


In [ ]:
# Criterion 1: baseline preprocessing (20 points).
assert callable(normalize_text)
for _text, _expected in [
    ("GREAT movie!!! 10/10", "great movie"),
    ("good-bad", "good bad"),
    ("  A\tGOOD\nmovie  ", "a good movie"),
    ("I didn't like it.", "i didn t like it"),
    ("Café MOVIE_2", "caf movie"),
    ("the movies were running", "the movies were running"),
    ("", ""),
    ("123 !!! \t", ""),
]:
    _actual = normalize_text(_text)
    assert isinstance(_actual, str)
    assert _actual == _expected, (_text, _actual, _expected)
print("Baseline preprocessing checks passed.")


## Supplied baseline instance

This cell creates the baseline using your `normalize_text`. Preserve its instance and settings. CountVectorizer uses default unigram counts and a default token pattern excluding one-character words. Your preprocessing applies during both fitting and prediction.


In [ ]:
baseline_pipeline = Pipeline([
    ("vectorizer", CountVectorizer(preprocessor=normalize_text)),
    ("classifier", LogisticRegression(max_iter=1000)),
])
_supplied_baseline = baseline_pipeline


## 2. Train and evaluate the baseline — 25 points

**Add a new code cell immediately below this cell.** Fit the supplied `baseline_pipeline` on `X_train, y_train`. Predict on `X_test` into `baseline_predictions` and calculate `baseline_accuracy` with `accuracy_score(y_test, baseline_predictions)`.

Store accuracy as a fraction and print it as a percentage. Do not replace or reconfigure the supplied pipeline, or fit on test data. No minimum accuracy is required.


In [ ]:
# Criterion 2: baseline training, predictions, and accuracy (25 points).
assert baseline_pipeline is _supplied_baseline
assert [name for name, _ in baseline_pipeline.steps] == ["vectorizer", "classifier"]
assert type(baseline_pipeline.named_steps["vectorizer"]) is CountVectorizer
assert baseline_pipeline.named_steps["vectorizer"].get_params() == CountVectorizer(preprocessor=normalize_text).get_params()
_check_fitted_pipeline(baseline_pipeline, baseline_predictions, baseline_accuracy)
print("Baseline training and evaluation checks passed.")


## Rules for both alternatives

Construct two **new, independent Pipeline instances**, each with its own new vectorizer and new `LogisticRegression(max_iter=1000)`. Keep other classifier settings at their defaults. Name the vectorizer step `"vectorizer"` and the final classifier step `"classifier"`; optional preprocessing transformer steps may come first.

Each design must differ from the baseline and the other alternative through preprocessing or vectorization. Choose CountVectorizer or TfidfVectorizer. **At least one alternative must use TF-IDF; at least one must add an advanced technique such as stemming, lemmatization, or n-grams.** TF-IDF alone can be the other experiment. Both NLTK and spaCy are installed and available for your choices; n-grams can use sklearn directly. If you use NLTK or spaCy, add setup cells that download any required corpora/models at runtime, before the cells that use them (`nltk.download(...)` or `spacy.cli.download(...)`). The notebook must run top to bottom without manual resource downloads beforehand. If your technique needs no downloaded resources, explain that in your notebook.

Keep processing inside the pipeline/vectorizer so it applies automatically to training, test, and future texts. Use exactly the supplied inputs and keep matrices sparse. Implement exactly two alternatives without search, cross-validation, or repeated test-driven tuning. Improvement is not required.

Add a brief markdown explanation for each alternative: changed techniques/settings, expected effect, and any additional resource setup. Structural checks support but do not replace teacher review of the advanced technique and explanations.


## 3. Alternative 1 — 20 points

**Add a new code cell immediately below this cell.** Construct `alternative_1_pipeline` from new components, fit it on `X_train, y_train`, and calculate `alternative_1_predictions` and `alternative_1_accuracy` on the supplied test set. Display accuracy as a percentage and add your markdown explanation in a new cell.


In [ ]:
# Criterion 3: first independent alternative, training, and evaluation (20 points).
_check_classifier(alternative_1_pipeline)
_check_independent(alternative_1_pipeline, [baseline_pipeline])
_check_fitted_pipeline(alternative_1_pipeline, alternative_1_predictions, alternative_1_accuracy)
print("Alternative 1 checks passed.")


## 4. Alternative 2 — 20 points

**Add a new code cell immediately below this cell.** Construct `alternative_2_pipeline` from new components, fit it on `X_train, y_train`, and calculate `alternative_2_predictions` and `alternative_2_accuracy` on the supplied test set. Display accuracy as a percentage and add your markdown explanation in a new cell.

It must differ from both previous designs. Across the two alternatives include TF-IDF and at least one advanced technique.


In [ ]:
# Criterion 4: second independent alternative, training, and evaluation (20 points).
_check_classifier(alternative_2_pipeline)
_check_independent(alternative_2_pipeline, [baseline_pipeline, alternative_1_pipeline])
assert any(
    isinstance(pipeline.named_steps["vectorizer"], TfidfVectorizer)
    for pipeline in (alternative_1_pipeline, alternative_2_pipeline)
), "At least one alternative must use TfidfVectorizer."
_check_fitted_pipeline(alternative_2_pipeline, alternative_2_predictions, alternative_2_accuracy)
print("Alternative 2 checks passed. The teacher also reviews the advanced technique.")


## Supplied future texts

These texts have no supplied true labels. Use the winning fitted pipeline to predict their sentiment; no specific prediction is required.


In [ ]:
new_texts = [
    "The acting was wonderful and I loved the story!",
    "This movie was boring, badly written, and far too long.",
    "I did not expect to enjoy it, but it was surprisingly good.",
    "Beautiful scenery could not save the disappointing plot.",
]
_fixed_new_texts = tuple(new_texts)


## 5. Compare and classify future texts — 15 points

**Add a new code cell immediately below this cell.** Create a pandas DataFrame `results` with exactly the columns `pipeline` and `accuracy` and three rows in order: `"baseline"`, `"alternative_1"`, `"alternative_2"`. Use their calculated accuracy fractions. Display percentages without changing stored values.

Store the highest-scoring name in `best_name` and its existing fitted instance in `best_pipeline`. Break ties by the table's original order. Call `best_pipeline.predict(new_texts)` into `new_predictions` and display each text and predicted sentiment. Do not refit or combine training and test data.

Add a brief markdown interpretation identifying the best observed classifier, whether alternatives helped, and one possible reason. This is a simplified comparison using test accuracy; the winning score is not a separate untouched final evaluation. No improvement or specific future-text prediction is required.


In [ ]:
# Criterion 5: comparison, winner selection, and future-text inference (15 points).
_check_fixed_inputs()
assert isinstance(results, pd.DataFrame)
assert list(results.columns) == ["pipeline", "accuracy"]
assert results.shape == (3, 2)
assert results["pipeline"].tolist() == ["baseline", "alternative_1", "alternative_2"]
_expected_accuracies = [baseline_accuracy, alternative_1_accuracy, alternative_2_accuracy]
np.testing.assert_allclose(results["accuracy"].to_numpy(), _expected_accuracies, rtol=0, atol=1e-12)
_names = ["baseline", "alternative_1", "alternative_2"]
_pipelines = [baseline_pipeline, alternative_1_pipeline, alternative_2_pipeline]
_expected_best_index = int(np.argmax(_expected_accuracies))
assert best_name == _names[_expected_best_index]
assert best_pipeline is _pipelines[_expected_best_index]
assert tuple(new_texts) == _fixed_new_texts
assert isinstance(new_predictions, np.ndarray)
assert new_predictions.shape == (len(new_texts),)
assert set(new_predictions.tolist()).issubset({0, 1})
np.testing.assert_array_equal(new_predictions, best_pipeline.predict(new_texts))
print("Comparison and future-text prediction checks passed.")
